In [19]:
# =======================================================================================
# UAS Media Trajectory & Visualization (EXIF + SRT Processing)
# =======================================================================================
# =======================================================================================
# Recursively processes a root directory and each subfolder independently to extract and
# integrate UAS geospatial data from:
#     - Images (GPS EXIF metadata)
#     - DJI video telemetry logs (SRT files)
#
# Outputs (per subfolder):
#     1. Interactive HTML web maps (Leaflet) with high-resolution satellite basemaps
#     2. KMZ files for Google Earth with color-coded trajectory visualization
#
# Key Features:
# - Recursive directory processing with independent outputs for each subfolder
# - Supports geotagged images and DJI SRT-based video trajectories
# - Automatic trajectory reconstruction from EXIF and SRT data sources
# - Clustered trajectory segmentation based on configurable time gaps (e.g., 2 minutes)
# - Flexible altitude handling:
#       • Absolute elevation (true Z from GPS)
#       • Clamp-to-ground visualization (for Google Earth display)
# - 3D trajectory support using onboard drone altitude (no external DEM required)
# - Auto-fit map extent to all trajectories within each processed folder
# - Consistent visualization across Leaflet web maps and Google Earth KMZ outputs
#
# =======================================================================================
# DISCLAIMER:
# Portions of this script were developed with the assistance of AI tools for code review,
# debugging, and testing. 
# =======================================================================================
# AUTHOR: Farid Javadnejad
# CREATED: 2026-02-24
# UPDATED: 2026-05-05
# =======================================================================================

In [20]:
# =======================================================================================
# PART 0: LIBRARIES
#=======================================================================================

# Import necessary libraries
import os
import re
import io
from tkinter import TRUE
import zipfile
from datetime import datetime
import numpy as np
from PIL import Image, ImageDraw, ExifTags
import simplekml
import folium
from folium import plugins

In [21]:
# ======================================================================================
# PART 1: CONFIGURATION
#=======================================================================================

# Root directory containing subfolders to process
from tkinter import FALSE


ROOT_FOLDER = r"C:\Users\USFJ139860\WSP O365\Southwest Geomatics - Business Development\Pursuits\2026\Kimley-Horn\Barcelona\CONTROL PHOTOS 260526"

# Elevation mode for outputs:
ELEVATION_MODE = 0   # 0: Clamped to Ground OR 1: Absolute

# Include image flight path trajectories in outputs (True/False). Video trajectories always included.
INCLUDE_IMAGE_TRAJECTORIES = FALSE

# Width of thumbnail images for embedded placemark images (pixels)
THUMBNAIL_WIDTH = 320

# Trajectory color palette (reused cyclically)
TRAJECTORY_COLORS = [
    '#FF0000', '#00FF00', '#0000FF', '#FFFF00',
    '#FF00FF', '#00FFFF', '#FF8800', '#8800FF']

# Line width for trajectory polylines in KMZ (video pipeline)
KMZ_LINE_WIDTH = 4

# Trajectory segmentation time gap (seconds) for image sequences
TIME_GAP_THRESHOLD = 10000  # 5 minutes


In [22]:
# ======================================================================================
# PART 2: IMAGE PIPELINE
#=======================================================================================
def convert_to_degrees(value):
    """Convert GPS (degrees, minutes, seconds) to decimal degrees."""
    try:
        d = float(value[0]); m = float(value[1]); s = float(value[2])
        return d + (m / 60.0) + (s / 3600.0)
    except Exception:
        return None

def parse_exif_datetime(datetime_str):
    """Parse EXIF DateTimeOriginal ('YYYY:MM:DD HH:MM:SS') into a datetime object."""
    try:
        return datetime.strptime(datetime_str, "%Y:%m:%d %H:%M:%S")
    except Exception:
        return None

def get_gps_data(image_path):
    """
    Extract GPS metadata from an image file. Returns a dict with:
    latitude, longitude, absolute_elevation, elevation_source,
    filename, timestamp, datetime_obj, trajectory_id.
    """
    info = {
        'latitude': None, 'longitude': None,
        'absolute_elevation': None, 'elevation_source': None,
        'filename': os.path.basename(image_path),
        'timestamp': None, 'datetime_obj': None, 'trajectory_id': None
    }
    try:
        with Image.open(image_path) as img:
            exif_data = img._getexif()
            if exif_data:
                gps_info = {}
                for tag, val in exif_data.items():
                    tag_name = ExifTags.TAGS.get(tag, tag)
                    if tag_name == 'GPSInfo':
                        for gps_tag in val:
                            gps_tag_name = ExifTags.GPSTAGS.get(gps_tag, gps_tag)
                            gps_info[gps_tag_name] = val[gps_tag]
                    if tag_name == 'DateTimeOriginal':
                        info['timestamp'] = str(val)
                        info['datetime_obj'] = parse_exif_datetime(str(val))
                # Process coordinates if present
                if 'GPSLatitude' in gps_info and 'GPSLongitude' in gps_info:
                    lat = convert_to_degrees(gps_info['GPSLatitude'])
                    lon = convert_to_degrees(gps_info['GPSLongitude'])
                    if lat is not None and lon is not None:
                        if gps_info.get('GPSLatitudeRef') == 'S':
                            lat = -lat
                        if gps_info.get('GPSLongitudeRef') == 'W':
                            lon = -lon
                        info['latitude'] = lat
                        info['longitude'] = lon
                        # Process altitude if available
                        if 'GPSAltitude' in gps_info:
                            try:
                                alt = gps_info['GPSAltitude']
                                alt_val = float(alt[0]) / float(alt[1]) if isinstance(alt, tuple) else float(alt)
                                info['absolute_elevation'] = alt_val
                                info['elevation_source'] = 'EXIF'
                            except Exception:
                                info['absolute_elevation'] = None
                                info['elevation_source'] = None
    except Exception as e:
        print(f"Error processing {image_path}: {e}")
    return info

def process_images(folder_path):
    """Scan a folder for JPEG images and extract GPS metadata from each. Return list of image metadata dicts."""
    image_data = []
    supported_exts = ('.jpg', '.jpeg', '.JPG', '.JPEG')
    print(f"Scanning folder: {folder_path}")
    if not os.path.exists(folder_path):
        print(f"ERROR: Folder not found: {folder_path}")
        return image_data
    for filename in os.listdir(folder_path):
        if filename.endswith(supported_exts):
            file_path = os.path.join(folder_path, filename)
            print(f"Processing: {filename}")
            meta = get_gps_data(file_path)
            if meta['latitude'] and meta['longitude']:
                meta['file_path'] = file_path
                image_data.append(meta)
                elev_str = f"{meta['absolute_elevation']:.1f} m ({meta['elevation_source']})" if meta['absolute_elevation'] is not None else "N/A"
                time_str = meta['timestamp'] or "N/A"
                print(f"  ✓ Lat: {meta['latitude']:.6f}, Lon: {meta['longitude']:.6f}, Elev: {elev_str}, Time: {time_str}")
            else:
                print(f"  ✗ No GPS data found")
    image_data.sort(key=lambda x: (x['datetime_obj'] is None, x['datetime_obj'], x['filename']))
    print(f"\nTotal images with GPS data: {len(image_data)}")
    exif_count = sum(1 for img in image_data if img['elevation_source'] == 'EXIF')
    none_count = sum(1 for img in image_data if img['elevation_source'] is None)
    print(f"Elevation sources: {exif_count} from EXIF, {none_count} missing")
    return image_data

def segment_trajectories(images, time_gap_seconds=TIME_GAP_THRESHOLD):
    """
    Divide images into separate trajectories if time gaps between successive images exceed threshold.
    Returns list of trajectory lists.
    """
    trajectories = []
    if not images:
        return trajectories
    current_traj = [images[0]]
    traj_id = 0
    images[0]['trajectory_id'] = traj_id
    for i in range(1, len(images)):
        prev_img = images[i-1]
        curr_img = images[i]
        if prev_img['datetime_obj'] and curr_img['datetime_obj']:
            time_diff = (curr_img['datetime_obj'] - prev_img['datetime_obj']).total_seconds()
            if time_diff > time_gap_seconds:
                trajectories.append(current_traj)
                traj_id += 1
                current_traj = [curr_img]
                curr_img['trajectory_id'] = traj_id
                print(f"  → New trajectory {traj_id} started: {time_diff:.0f}s gap between {prev_img['filename']} and {curr_img['filename']}")
            else:
                curr_img['trajectory_id'] = traj_id
                current_traj.append(curr_img)
        else:
            curr_img['trajectory_id'] = traj_id
            current_traj.append(curr_img)
            if not curr_img['datetime_obj']:
                print(f"  ⚠ No timestamp for {curr_img['filename']}, grouped by file order")
    trajectories.append(current_traj)
    print(f"\nSegmented into {len(trajectories)} trajectories:")
    for idx, traj in enumerate(trajectories):
        color = TRAJECTORY_COLORS[idx % len(TRAJECTORY_COLORS)]
        print(f"  Segment {idx+1}: {len(traj)} images, Color: {color}")
    return trajectories

def create_circle_icon(color, size=40):
    """Generate a circular PNG icon of a given color (with white outline) and return its bytes."""
    img = Image.new('RGBA', (size, size), (0, 0, 0, 0))
    draw = ImageDraw.Draw(img)
    draw.ellipse([2, 2, size-2, size-2], fill=color, outline='white', width=3)
    center = size // 2
    draw.ellipse([center-3, center-3, center+3, center+3], fill='white', outline=color, width=2)
    buf = io.BytesIO()
    img.save(buf, format='PNG')
    return buf.getvalue()

def create_leaflet_map(image_data, trajectories, output_file):
    """Generate an interactive Leaflet HTML map for image locations (and optional trajectories)."""
    if not image_data:
        print("No data to map!")
        return
    all_lats = [img['latitude'] for img in image_data]
    all_lons = [img['longitude'] for img in image_data]
    avg_lat = sum(all_lats)/len(all_lats)
    avg_lon = sum(all_lons)/len(all_lons)
    # Initialize map centered on average location
    m = folium.Map(location=[avg_lat, avg_lon], zoom_start=16, tiles=None)
    folium.TileLayer(
        tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
        attr='Esri', name='Esri Satellite', overlay=False, control=True, show=True
    ).add_to(m)
    folium.TileLayer(
        tiles='https://services.arcgisonline.com/ArcGIS/rest/services/Reference/World_Boundaries_and_Places/MapServer/tile/{z}/{y}/{x}',
        attr='Esri', name='Esri Labels', overlay=True, control=False, show=True
    ).add_to(m)
    folium.TileLayer(
        tiles='OpenStreetMap', name='Street Map', overlay=False, control=True, show=False
    ).add_to(m)
    # Add markers for each image
    for img in image_data:
        traj_id = img.get('trajectory_id', 0)
        color = TRAJECTORY_COLORS[traj_id % len(TRAJECTORY_COLORS)]
        elev_value = f"{img['absolute_elevation']:.1f} m" if img['absolute_elevation'] is not None else "N/A"
        # Determine elevation source text for display
        if ELEVATION_MODE == 0:
            elev_source_text = "Clamped to Ground"
        else:
            elev_source_text = "EXIF GPSAltitude (AMSL)" if img['elevation_source'] == 'EXIF' else "N/A (Clamped to Ground)"
        popup_html = f"""
        <div style="width: 300px;">
            <h4 style="margin-bottom: 8px;">{img['filename']}</h4>
            <table style="width:100%; font-size: 12px;">
                <tr><td><b>Latitude:</b></td><td>{img['latitude']:.6f}°</td></tr>
                <tr><td><b>Longitude:</b></td><td>{img['longitude']:.6f}°</td></tr>
                <tr><td><b>Altitude (AMSL):</b></td><td>{elev_value}</td></tr>
                <tr><td><b>Elevation Source:</b></td><td>{elev_source_text}</td></tr>
                <tr><td><b>Segment:</b></td><td>{traj_id + 1}</td></tr>
                <tr><td><b>Segment Color:</b></td>
                    <td><span style="background-color:{color}; padding:2px 10px; color:white;">&nbsp;&nbsp;&nbsp;</span></td>
                </tr>
                <tr><td><b>Timestamp:</b></td><td>{img['timestamp'] or 'N/A'}</td></tr>
            </table>
        </div>
        """
        folium.CircleMarker(
            location=[img['latitude'], img['longitude']],
            radius=8,
            popup=folium.Popup(popup_html, max_width=THUMBNAIL_WIDTH),
            tooltip=f"{img['filename']} (Segment {traj_id + 1})",
            color='white', fillColor=color, fillOpacity=0.8, weight=2
        ).add_to(m)
    # Add trajectory polylines if enabled
    if INCLUDE_IMAGE_TRAJECTORIES:
        for traj_idx, traj in enumerate(trajectories):
            if len(traj) > 1:
                coords = [(p['latitude'], p['longitude']) for p in traj]
                traj_color = TRAJECTORY_COLORS[traj_idx % len(TRAJECTORY_COLORS)]
                folium.PolyLine(coords, color=traj_color, weight=3, opacity=0.8,
                                name=f'Segment {traj_idx+1} ({len(traj)} images)').add_to(m)
    m.fit_bounds([[min(all_lats), min(all_lons)], [max(all_lats), max(all_lons)]], padding=[30, 30])
    folium.LayerControl(position='topright').add_to(m)
    plugins.Fullscreen().add_to(m)
    plugins.MeasureControl(position='topleft').add_to(m)
    m.save(output_file)
    print(f"\n✓ Interactive map saved: {output_file}")
    print("  - Default basemap: Esri Satellite + Labels")
    print("  - Auto-zoomed to extent of all images and trajectories")

def create_kmz_file_images(image_data, trajectories, output_file):
    """
    Create a KMZ file with placemarks for images (embedded thumbnails) and optional flight path lines.
    """
    if not image_data:
        print("No data to export to KMZ!")
        return
    kml = simplekml.Kml()
    # Prepare icon images for trajectory markers (circle icons with different colors)
    icon_files = {}
    num_traj = len(trajectories)
    for i in range(num_traj):
        color = TRAJECTORY_COLORS[i % len(TRAJECTORY_COLORS)]
        icon_name = f'icon_trajectory_{i+1}.png'
        icon_files[icon_name] = create_circle_icon(color, size=40)
    thumb_files = {}
    points_folder = kml.newfolder(name='Image Locations')
    for img in image_data:
        pnt = points_folder.newpoint(name=img['filename'])
        # Set coordinates and altitude mode based on ELEVATION_MODE
        if ELEVATION_MODE == 0:
            pnt.coords = [(img['longitude'], img['latitude'])]
            pnt.altitudemode = simplekml.AltitudeMode.clamptoground
        else:
            alt_val = img['absolute_elevation'] if img['absolute_elevation'] is not None else 0
            pnt.coords = [(img['longitude'], img['latitude'], alt_val)]
            pnt.altitudemode = simplekml.AltitudeMode.absolute
            pnt.extrude = 1  # tether line from marker to ground
        # Assign colored icon for the placemark
        traj_id = img.get('trajectory_id', 0)
        icon_href = f'icon_trajectory_{(traj_id % len(TRAJECTORY_COLORS)) + 1}.png'
        pnt.style.iconstyle.icon.href = icon_href
        pnt.style.iconstyle.scale = 1.0
        pnt.style.iconstyle.hotspot = simplekml.HotSpot(x=0.5, y=0.5, xunits='fraction', yunits='fraction')
        # Generate a thumbnail image (320px width) for this placemark's balloon
        thumb_name = f"{os.path.splitext(img['filename'])[0]}_thumb.jpg"
        if thumb_name not in thumb_files:
            try:
                from PIL import Image
                import io
                
                # EXIF Orientation to Pillow transpose mapping
                ORIENTATION_TRANSPOSES = {
                    1: None,               # Normal - no rotation
                    3: Image.ROTATE_180,   # Rotated 180°
                    6: Image.ROTATE_270,   # Rotated 90° CW → rotate 270° CCW to fix
                    8: Image.ROTATE_90,    # Rotated 270° CW → rotate 90° CCW to fix
                }
                
                with Image.open(img['file_path']) as im:
                    # Step 1: Read EXIF orientation (minimal EXIF access)
                    orientation = 1  # Default: normal
                    try:
                        exif = im.getexif()
                        if exif:
                            orientation = exif.get(0x0112, 1)  # 0x0112 = Orientation tag
                    except:
                        pass
                    
                    # Step 2: Physically rotate pixels if needed
                    transpose_op = ORIENTATION_TRANSPOSES.get(orientation, None)
                    if transpose_op is not None:
                        im = im.transpose(transpose_op)
                    
                    # Step 3: Convert and resize (pixels now correctly oriented)
                    im = im.convert('RGB')
                    w, h = im.size
                    new_w = THUMBNAIL_WIDTH
                    new_h = int(h * new_w / w) if w > 0 else h
                    im_thumb = im.resize((new_w, new_h), Image.LANCZOS)
                    
                    # Step 4: Save WITHOUT EXIF
                    buf = io.BytesIO()
                    im_thumb.save(buf, format='JPEG', quality=85, exif=b'')
                    thumb_data = buf.getvalue()
                    
                    if len(thumb_data) > 200 * 1024:
                        buf = io.BytesIO()
                        im_thumb.save(buf, format='JPEG', quality=75, exif=b'')
                        thumb_data = buf.getvalue()
                    
                    thumb_files[thumb_name] = thumb_data
                    
            except Exception as e:
                print(f"⚠ Could not create thumbnail for {img['filename']}: {e}")
                thumb_name = None

        # Build placemark description with thumbnail and metadata table
        elev_value = f"{img['absolute_elevation']:.1f} m" if img['absolute_elevation'] is not None else "N/A"
        if ELEVATION_MODE == 0:
            elev_source_text = "Clamped to Ground"
        else:
            elev_source_text = "Absolute" if img['elevation_source'] == 'EXIF' else "Clamped to Ground"
        # *** FIXED: Add proper <img> tag ***
        thumb_tag = f'<img src="{thumb_name}" width="{THUMBNAIL_WIDTH}" alt="{img["filename"]}"/><br/><br/>' if thumb_name else ""
        
        pnt.description = f"""<![CDATA[
        <h4>{img['filename']}</h4>
        {thumb_tag}
        <table border="0" cellpadding="5" style="margin-top:5px;">
            <tr><th>Property</th><th>Value</th></tr>
            <tr><td>Latitude</td><td>{img['latitude']:.6f}°</td></tr>
            <tr><td>Longitude</td><td>{img['longitude']:.6f}°</td></tr>
            <tr><td>Elevation (m)</td><td>{elev_value}</td></tr>
            <tr><td>Altitude Mode</td><td>{elev_source_text}</td></tr>
            <tr><td>Segment</td><td>{traj_id + 1}</td></tr>
            <tr><td>Segment Color</td>
                <td><span style="background-color:{TRAJECTORY_COLORS[traj_id % len(TRAJECTORY_COLORS)]}; padding:2px 10px; color:white;">&nbsp;&nbsp;&nbsp;</span></td>
            </tr>
            <tr><td>Timestamp</td><td>{img['timestamp'] or 'N/A'}</td></tr>
        </table>
        ]]>
        """
    # Add flight path lines if enabled
    if INCLUDE_IMAGE_TRAJECTORIES and trajectories:
        paths_folder = kml.newfolder(name='Flight Paths')
        for idx, traj in enumerate(trajectories):
            if len(traj) > 1:
                coords = []
                for p in traj:
                    alt_val = p['absolute_elevation'] if p['absolute_elevation'] is not None else 0
                    coords.append((p['longitude'], p['latitude'], alt_val))
                line = paths_folder.newlinestring(name="")
                line.coords = coords
                hex_color = TRAJECTORY_COLORS[idx % len(TRAJECTORY_COLORS)].lstrip('#')
                line.style.linestyle.color = f'ff{hex_color[4:6]}{hex_color[2:4]}{hex_color[0:2]}'
                line.style.linestyle.width = 4
                line.altitudemode = simplekml.AltitudeMode.clamptoground if ELEVATION_MODE == 0 else simplekml.AltitudeMode.absolute
                # Optional trajectory summary in description
                start_time = traj[0]['timestamp'] or 'N/A'
                end_time = traj[-1]['timestamp'] or 'N/A'
                alt_list = [p['absolute_elevation'] for p in traj if p['absolute_elevation'] is not None]
                avg_elev = f"{np.mean(alt_list):.1f}" if alt_list else "0"
                min_elev = f"{min(alt_list):.1f}" if alt_list else "0"
                max_elev = f"{max(alt_list):.1f}" if alt_list else "0"
                elev_src_note = "Clamped to ground in output" if ELEVATION_MODE == 0 else \
                                ("Elevation Source: EXIF GPSAltitude (AMSL)" if alt_list else "Elevation Source: N/A (Clamped to Ground)")
                line.description = f"""
                <![CDATA[
                <b>Segment {idx+1}</b><br/>
                Images: {len(traj)}<br/>
                Average Altitude: {avg_elev} m<br/>
                Altitude Range: {min_elev} - {max_elev} m<br/>
                Start: {start_time}<br/>
                End: {end_time}<br/>{elev_src_note}
                ]]>
                """
    # Save KML and embed images into KMZ archive
    temp_kml = output_file.replace('.kmz', '_temp.kml')
    kml.save(temp_kml)
    with zipfile.ZipFile(output_file, 'w', zipfile.ZIP_DEFLATED) as kmz:
        kmz.write(temp_kml, 'doc.kml')
        for fname, data in {**icon_files, **thumb_files}.items():
            kmz.writestr(fname, data)
    os.remove(temp_kml)
    print(f"✓ KMZ file saved: {output_file} (thumbnails embedded)")

In [23]:
# ======================================================================================
# PART 3: IMAGE PIPELINE
#=======================================================================================

try:
    import cv2
    OPENCV_AVAILABLE = True
except ImportError:
    OPENCV_AVAILABLE = False

def parse_dji_srt_line(line):
    """Extract latitude, longitude, altitudes, and timestamp from one line of a DJI SRT file."""
    point = {'latitude': None, 'longitude': None, 'abs_altitude': None, 'rel_altitude': None, 'timestamp': None}
    try:
        lat_match = re.search(r'\[latitude:\s*([-\d.]+)\]', line)
        if lat_match:
            point['latitude'] = float(lat_match.group(1))
        lon_match = re.search(r'\[longitude:\s*([-\d.]+)\]', line)
        if lon_match:
            point['longitude'] = float(lon_match.group(1))
        abs_alt_match = re.search(r'abs_alt:\s*([-\d.]+)', line)
        if abs_alt_match:
            point['abs_altitude'] = float(abs_alt_match.group(1))
        rel_alt_match = re.search(r'rel_alt:\s*([-\d.]+)', line)
        if rel_alt_match:
            point['rel_altitude'] = float(rel_alt_match.group(1))
        ts_match = re.search(r'(\d{4}-\d{2}-\d{2}\s+\d{2}:\d{2}:\d{2}\.\d{3})', line)
        if ts_match:
            point['timestamp'] = ts_match.group(1)
    except Exception:
        pass
    return point

def parse_srt_file(srt_path):
    """Parse an entire DJI SRT file and return a list of trajectory points dicts."""
    points = []
    try:
        with open(srt_path, 'r', encoding='utf-8', errors='ignore') as file:
            for line in file:
                pt = parse_dji_srt_line(line)
                if pt['latitude'] is not None and pt['longitude'] is not None:
                    points.append(pt)
    except Exception as e:
        print(f"Error parsing {os.path.basename(srt_path)}: {e}")
    return points

def process_srt_files(folder_path):
    """
    Scan a folder for .srt files (DJI SRT telemetry) and extract trajectories.
    Returns list of video trajectory dicts (filename, points list, color, video_id).
    """
    video_trajectories = []
    print(f"Scanning folder: {folder_path}")
    if not os.path.exists(folder_path):
        print(f"ERROR: Folder not found: {folder_path}")
        return video_trajectories
    srt_files = [f for f in os.listdir(folder_path) if f.lower().endswith('.srt')]
    if not srt_files:
        print(f"WARNING: No .srt files found in {folder_path}")
        return video_trajectories
    print(f"Found {len(srt_files)} .srt file(s)")
    for idx, filename in enumerate(sorted(srt_files)):
        file_path = os.path.join(folder_path, filename)
        print(f"\nProcessing: {filename}")
        points = parse_srt_file(file_path)
        if points:
            color = TRAJECTORY_COLORS[idx % len(TRAJECTORY_COLORS)]
            video_trajectories.append({
                'filename': filename,
                'points': points,
                'color': color,
                'video_id': idx
            })
            print(f"  ✓ Extracted {len(points)} GPS points")
            if points:
                print(f"  ✓ Start: Lat {points[0]['latitude']:.6f}, Lon {points[0]['longitude']:.6f}")
                print(f"  ✓ End:   Lat {points[-1]['latitude']:.6f}, Lon {points[-1]['longitude']:.6f}")
            print(f"  ✓ Assigned color: {color}")
        else:
            print(f"  ✗ No valid GPS points found in {filename}")
    print(f"\nTotal videos with trajectories: {len(video_trajectories)}")
    return video_trajectories

def create_trajectory_map(video_trajectories, output_file):
    """Create a Leaflet HTML map for video trajectories."""
    if not video_trajectories:
        print("No trajectories to map!")
        return
    all_lats = [p['latitude'] for traj in video_trajectories for p in traj['points']]
    all_lons = [p['longitude'] for traj in video_trajectories for p in traj['points']]
    m = folium.Map(location=[sum(all_lats)/len(all_lats), sum(all_lons)/len(all_lons)], zoom_start=16, tiles=None)
    folium.TileLayer(
        tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
        attr='Esri', name='Esri Satellite', overlay=False, control=True, show=True
    ).add_to(m)
    folium.TileLayer(
        tiles='https://services.arcgisonline.com/ArcGIS/rest/services/Reference/World_Boundaries_and_Places/MapServer/tile/{z}/{y}/{x}',
        attr='Esri', name='Esri Labels', overlay=True, control=True, show=True
    ).add_to(m)
    folium.TileLayer(
        tiles='OpenStreetMap', name='Street Map', overlay=False, control=True, show=False
    ).add_to(m)
    for traj in video_trajectories:
        color = traj['color']
        filename = traj['filename']
        points = traj['points']
        coords = [(p['latitude'], p['longitude']) for p in points]
        start_pt = points[0]; end_pt = points[-1]
        # Compute altitude stats across the trajectory
        alt_values = [p['abs_altitude'] for p in points if p['abs_altitude'] is not None]
        if not alt_values:
            alt_values = [p['rel_altitude'] for p in points if p['rel_altitude'] is not None]
        avg_alt = sum(alt_values)/len(alt_values) if alt_values else 0
        min_alt = min(alt_values) if alt_values else 0
        max_alt = max(alt_values) if alt_values else 0
        elev_source_text = "Clamped to Ground" if ELEVATION_MODE == 0 else \
                           ("Video SRT Telemetry (AMSL)" if any(p['abs_altitude'] is not None for p in points) else "N/A (Clamped to Ground)")
        folium.PolyLine(
            coords, color=color, weight=3, opacity=0.8,
            name=f'{filename} ({len(points)} points)',
            popup=folium.Popup(f"""
                <div style="width: 280px;">
                    <h4 style="margin-bottom: 8px;">{filename}</h4>
                    <table style="width:100%; font-size: 12px;">
                        <tr><td><b>Total Points:</b></td><td>{len(points)}</td></tr>
                        <tr><td><b>Avg Altitude:</b></td><td>{avg_alt:.1f} m</td></tr>
                        <tr><td><b>Altitude Range:</b></td><td>{min_alt:.1f} - {max_alt:.1f} m</td></tr>
                        <tr><td><b>Elevation Source:</b></td><td>{elev_source_text}</td></tr>
                        <tr><td><b>Start Time:</b></td><td>{start_pt['timestamp'] or 'N/A'}</td></tr>
                        <tr><td><b>End Time:</b></td><td>{end_pt['timestamp'] or 'N/A'}</td></tr>
                    </table>
                </div>
            """, max_width=300)
        ).add_to(m)
        folium.CircleMarker(
            location=[start_pt['latitude'], start_pt['longitude']],
            radius=10,
            popup=folium.Popup(f"""
                <div style="width: 250px;">
                    <h4 style="color: {color};">▶ START: {filename}</h4>
                    <table style="width:100%; font-size: 12px;">
                        <tr><td><b>Latitude:</b></td><td>{start_pt['latitude']:.6f}°</td></tr>
                        <tr><td><b>Longitude:</b></td><td>{start_pt['longitude']:.6f}°</td></tr>
                        <tr><td><b>Altitude:</b></td><td>{(start_pt.get('abs_altitude') or start_pt.get('rel_altitude') or 0):.1f} m</td></tr>
                        <tr><td><b>Elevation Source:</b></td><td>{elev_source_text}</td></tr>
                        <tr><td><b>Timestamp:</b></td><td>{start_pt['timestamp'] or 'N/A'}</td></tr>
                    </table>
                </div>
            """, max_width=270),
            tooltip=f"▶ START: {filename}",
            color='white', fillColor=color, fillOpacity=0.9, weight=4
        ).add_to(m)
        folium.CircleMarker(
            location=[end_pt['latitude'], end_pt['longitude']],
            radius=10,
            popup=folium.Popup(f"""
                <div style="width: 250px;">
                    <h4 style="color: {color};">■ END: {filename}</h4>
                    <table style="width:100%; font-size: 12px;">
                        <tr><td><b>Latitude:</b></td><td>{end_pt['latitude']:.6f}°</td></tr>
                        <tr><td><b>Longitude:</b></td><td>{end_pt['longitude']:.6f}°</td></tr>
                        <tr><td><b>Altitude:</b></td><td>{(end_pt.get('abs_altitude') or end_pt.get('rel_altitude') or 0):.1f} m</td></tr>
                        <tr><td><b>Elevation Source:</b></td><td>{elev_source_text}</td></tr>
                        <tr><td><b>Timestamp:</b></td><td>{end_pt['timestamp'] or 'N/A'}</td></tr>
                    </table>
                </div>
            """, max_width=270),
            tooltip=f"■ END: {filename}",
            color='white', fillColor=color, fillOpacity=0.9, weight=4
        ).add_to(m)
    m.fit_bounds([[min(all_lats), min(all_lons)], [max(all_lats), max(all_lons)]], padding=[30, 30])
    folium.LayerControl(position='topright').add_to(m)
    plugins.Fullscreen().add_to(m)
    plugins.MeasureControl(position='topleft').add_to(m)
    m.save(output_file)
    print(f"\n✓ Interactive map saved: {output_file}")
    print("  - Default basemap: Esri Satellite + Labels")
    print("  - Auto-zoomed to extent of all video trajectories")

def create_kmz_file_videos(video_trajectories, output_file):
    """
    Create a KMZ file with a line for each video trajectory (no thumbnails).
    """
    if not video_trajectories:
        print("No trajectories to export to KMZ!")
        return
    
    kml = simplekml.Kml()
    
    for traj in video_trajectories:
        filename = traj['filename']
        base_name = os.path.splitext(filename)[0]
        
        # Create trajectory line in KML
        coords = []
        for p in traj['points']:
            lat, lon = p['latitude'], p['longitude']
            if ELEVATION_MODE == 1:
                alt_val = p['abs_altitude'] if p['abs_altitude'] is not None else (p['rel_altitude'] or 0)
            else:
                alt_val = 0
            coords.append((lon, lat, alt_val))
        
        line = kml.newlinestring(name=base_name)
        line.coords = coords
        hex_color = traj['color'].lstrip('#')
        line.style.linestyle.color = f'ff{hex_color[4:6]}{hex_color[2:4]}{hex_color[0:2]}'
        line.style.linestyle.width = KMZ_LINE_WIDTH
        line.altitudemode = simplekml.AltitudeMode.clamptoground if ELEVATION_MODE == 0 else simplekml.AltitudeMode.absolute
        
        # Add line description with stats (no thumbnail)
        alt_list = [p['abs_altitude'] for p in traj['points'] if p['abs_altitude'] is not None]
        if not alt_list:
            alt_list = [p['rel_altitude'] for p in traj['points'] if p['rel_altitude'] is not None]
        min_alt = f"{min(alt_list):.1f}" if alt_list else "N/A"
        max_alt = f"{max(alt_list):.1f}" if alt_list else "N/A"
        elev_source_text = "Clamped to Ground" if ELEVATION_MODE == 0 else \
                           ("Absolute" if any(p['abs_altitude'] is not None for p in traj['points']) else "Clamped to Ground")
        
        line.description = f"""<![CDATA[
        <h4>{base_name}</h4>
        <p><b>Points:</b> {len(traj['points'])}<br/>
           <b>Altitude Range:</b> {min_alt} - {max_alt} m<br/>
           <b>Elevation Source:</b> {elev_source_text}<br/>
           <b>Start:</b> {traj['points'][0]['timestamp'] or 'N/A'}<br/>
           <b>End:</b> {traj['points'][-1]['timestamp'] or 'N/A'}</p>
        ]]>
        """
    
    # Save KML (no thumbnails to embed)
    temp_kml = output_file.replace('.kmz', '_temp.kml')
    kml.save(temp_kml)
    with zipfile.ZipFile(output_file, 'w', zipfile.ZIP_DEFLATED) as kmz:
        kmz.write(temp_kml, 'doc.kml')
    os.remove(temp_kml)
    print(f"✓ KMZ file saved: {output_file} (video trajectories, no thumbnails)")

In [24]:
# ======================================================================================
# PART 4: MAIN
#=======================================================================================

def run_image_processing(folder_path):
    """Process a folder of images and generate outputs."""
    print("=" * 70)
    print("DRONE IMAGE GEOTAG EXTRACTOR - TRAJECTORY-BASED COLORING")
    print("=" * 70)
    image_data = process_images(folder_path)
    if not image_data:
        print("\n⚠ No images with GPS data found. Please check:")
        print("  1. Folder path is correct")
        print("  2. Images contain EXIF GPS metadata")
        print("  3. Images are in JPEG format")
        return
    print("\n" + "=" * 70)
    print(f"SEGMENTING TRAJECTORIES ({TIME_GAP_THRESHOLD//60}-minute threshold)")
    print("=" * 70)
    trajectories = segment_trajectories(image_data, TIME_GAP_THRESHOLD)
    output_html = os.path.join(folder_path, "image_locations.html")
    output_kmz = os.path.join(folder_path, "image_locations.kmz")
    print("\n" + "=" * 70)
    print("GENERATING OUTPUTS")
    print("=" * 70)
    create_leaflet_map(image_data, trajectories, output_html)
    create_kmz_file_images(image_data, trajectories, output_kmz)
    print("\n" + "=" * 70)
    print("PROCESS COMPLETE!")
    print("=" * 70)
    print(f"\nOutputs created in: {folder_path}")
    print(f"  • HTML Map: {output_html}")
    print(f"  • KMZ File: {output_kmz}")
    print(f"\nTotal images processed: {len(image_data)}")
    print(f"Total trajectories: {len(trajectories)}")
    print(f"Elevation Mode: {ELEVATION_MODE}, Image trajectories included: {INCLUDE_IMAGE_TRAJECTORIES}")

def run_video_processing(folder_path):
    """Process a folder of DJI video SRT files and generate outputs."""
    print("=" * 70)
    print("DJI VIDEO SRT TRAJECTORY MAPPER")
    print("=" * 70)
    video_trajs = process_srt_files(folder_path)
    if not video_trajs:
        print("\n⚠ No video trajectories found. Please check:")
        print("  1. Folder path is correct")
        print("  2. Folder contains .srt files")
        print("  3. SRT files are in DJI format with GPS data")
        return
    print("\n" + "=" * 70)
    print("GENERATING OUTPUTS")
    print("=" * 70)
    output_html = os.path.join(folder_path, "video_trajectories.html")
    output_kmz = os.path.join(folder_path, "video_trajectories.kmz")
    create_trajectory_map(video_trajs, output_html)
    create_kmz_file_videos(video_trajs, output_kmz)
    print("\n" + "=" * 70)
    print("PROCESS COMPLETE!")
    print("=" * 70)
    print(f"\nOutputs created in: {folder_path}")
    print(f"  • HTML Map: {output_html}")
    print(f"  • KMZ File: {output_kmz}")
    print(f"\nTotal videos processed: {len(video_trajs)}")
    print("\Segment colors:")
    for traj in video_trajs:
        print(f"  • {traj['filename']}: {traj['color']}")

def main():
    if not os.path.exists(ROOT_FOLDER):
        print(f"ERROR: Specified root folder not found: {ROOT_FOLDER}")
        return
    for dirpath, dirnames, filenames in os.walk(ROOT_FOLDER):
        if dirpath == ROOT_FOLDER and not filenames:
            continue
        has_images = any(fname.lower().endswith(('.jpg', '.jpeg')) for fname in filenames)
        has_srts = any(fname.lower().endswith('.srt') for fname in filenames)
        if not has_images and not has_srts:
            continue
        if has_images:
            run_image_processing(dirpath)
        if has_srts:
            run_video_processing(dirpath)
        if has_images or has_srts:
            print()  # blank line between different folder outputs

if __name__ == "__main__":
    main()

<>:64: SyntaxWarning: invalid escape sequence '\S'
<>:64: SyntaxWarning: invalid escape sequence '\S'
C:\Users\USFJ139860\AppData\Local\Temp\ipykernel_54316\2125688997.py:64: SyntaxWarning: invalid escape sequence '\S'
  print("\Segment colors:")


DRONE IMAGE GEOTAG EXTRACTOR - TRAJECTORY-BASED COLORING
Scanning folder: C:\Users\USFJ139860\WSP O365\Southwest Geomatics - Business Development\Pursuits\2026\Kimley-Horn\Barcelona\CONTROL PHOTOS 260526
Processing: IMG_1909.jpeg
  ✓ Lat: 35.028836, Lon: -106.706092, Elev: 1502.0 m (EXIF), Time: 2026:05:26 10:28:49
Processing: IMG_1910.jpeg
  ✓ Lat: 35.028808, Lon: -106.706100, Elev: 1502.1 m (EXIF), Time: 2026:05:26 10:28:55
Processing: IMG_1911.jpeg
  ✓ Lat: 35.030364, Lon: -106.700508, Elev: 1484.7 m (EXIF), Time: 2026:05:26 10:37:57
Processing: IMG_1912.jpeg
  ✓ Lat: 35.030364, Lon: -106.700508, Elev: 1484.7 m (EXIF), Time: 2026:05:26 10:37:59
Processing: IMG_1913.jpeg
  ✓ Lat: 35.030117, Lon: -106.700531, Elev: 1503.9 m (EXIF), Time: 2026:05:26 10:40:04
Processing: IMG_1914.jpeg
  ✓ Lat: 35.030033, Lon: -106.700439, Elev: 1503.9 m (EXIF), Time: 2026:05:26 10:40:09
Processing: IMG_1915.jpeg
  ✓ Lat: 35.029461, Lon: -106.700189, Elev: 1503.9 m (EXIF), Time: 2026:05:26 10:42:50
Proce